# ML-06 — Signal Audit: Do the Flags Hold?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Vishal-141206/flyrank-ml-internship/blob/main/work/notebooks/w04_signal_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
# --- Imports ---
import os
from pathlib import Path

import duckdb
import numpy as np
import pandas as pd
from IPython.display import display

# --- Token loading (never prints the token) ---
def _load_hf_token():
    if os.environ.get("HF_TOKEN"):
        return os.environ["HF_TOKEN"]
    for p in (Path(".env"), Path("../.env"), Path("../../.env")):
        if p.exists():
            for line in p.read_text().splitlines():
                line = line.strip()
                if line.startswith("HF_TOKEN="):
                    return line.split("=", 1)[1].strip()
    try:
        from google.colab import userdata
        return userdata.get("HF_TOKEN")
    except Exception:
        return None

token = _load_hf_token()
assert token, "HF_TOKEN not found — check .env or environment"

con = duckdb.connect()
con.execute(f"CREATE SECRET hf_secret (TYPE huggingface, TOKEN '{token}')")

BASE = "hf://datasets/FlyRank/internship-warehouse"

def _ensure(name, sql):
    if con.execute("SELECT 1 FROM information_schema.tables WHERE table_name=?", [name]).fetchone():
        return
    con.execute(f"CREATE TEMP TABLE {name} AS {sql}")
    print(f"cached {name}")

_ensure("mar", f"SELECT * EXCLUDE (month) FROM read_parquet('{BASE}/fact_content_daily_performance/month=2026-03/*.parquet', hive_partitioning=true)")
_ensure("apr", f"SELECT * EXCLUDE (month) FROM read_parquet('{BASE}/fact_content_daily_performance/month=2026-04/*.parquet', hive_partitioning=true)")
_ensure("dim_clients", f"SELECT * FROM read_parquet('{BASE}/dim_clients.parquet')")

cached mar
cached apr
cached dim_clients


In [2]:
con.execute(f"CREATE TEMP TABLE dim_content AS SELECT * FROM read_parquet('{BASE}/dim_content.parquet')")
print("cached dim_content")

cached dim_content


In [3]:
feat = con.execute("""
SELECT
  content_hash_id,
  client_hash_id,
  SUM(gsc_impressions) FILTER (WHERE gsc_data_available IS TRUE)                            AS gsc_impressions_total,
  SUM(gsc_clicks)     FILTER (WHERE gsc_data_available IS TRUE)                             AS gsc_clicks_total,
  COUNT(*)           FILTER (WHERE gsc_data_available IS TRUE)                              AS gsc_active_days,
  SUM(gsc_impressions * gsc_avg_position)
    FILTER (WHERE gsc_data_available IS TRUE AND gsc_avg_position > 0)
    / NULLIF(SUM(gsc_impressions)
        FILTER (WHERE gsc_data_available IS TRUE AND gsc_avg_position > 0), 0)              AS gsc_avg_position_w
FROM mar
WHERE gsc_data_available IS TRUE
GROUP BY 1, 2
""").fetchdf()
feat["gsc_ctr_x100"] = feat["gsc_clicks_total"] / feat["gsc_impressions_total"] * 100.0

content_meta = con.execute("SELECT content_hash_id, content_type FROM dim_content").fetchdf()
feat = feat.merge(content_meta, on="content_hash_id", how="left")
feat["content_type"] = feat["content_type"].fillna("unknown")

print(f"frame shape: {feat.shape}")
feat.head()

frame shape: (176738, 8)


,content_hash_id,client_hash_id,gsc_impressions_total,gsc_clicks_total,gsc_active_days,gsc_avg_position_w,gsc_ctr_x100,content_type
0,content_4dc9bb9b3a416cd8,client_62f4a7e64f5e0096,97.0,0.0,23,13.821053,0.000000,keyword article
1,content_8cee10b4c91d2ab9,client_62f4a7e64f5e0096,844.0,0.0,31,2.179122,0.000000,keyword article
2,content_ae2bcea576f57235,client_62f4a7e64f5e0096,116.0,0.0,28,4.938053,0.000000,keyword article
3,content_c312bf288d781c2c,client_9958f0a7ae1df715,265.0,1.0,31,15.579545,0.377358,keyword article
4,content_c80a4531a5d452cd,client_9958f0a7ae1df715,1003.0,0.0,31,9.957129,0.000000,keyword article


## 1. Distributions

*Look before deciding: distributions of your key fields. Note the heavy tails.*

Before testing any signal, look at the shape of the key fields — heavy-tailed data (a few
huge values, a long tail of small ones) breaks assumptions like "the mean is a typical value"
and can make a signal look stronger or weaker than it really is if not accounted for.

Using the March 2026 feature frame built in ML-05 (`feat`): impressions and clicks are the
classic heavy-tail suspects in search data (a handful of pages dominate traffic), so I check
those first alongside position and CTR.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
numeric_cols = ["gsc_impressions_total", "gsc_clicks_total", "gsc_active_days", "gsc_avg_position_w", "gsc_ctr_x100"]

# Standard describe() first
display(feat[numeric_cols].describe().T)

# Heavy-tail check: compare mean vs median, and look at the top 1%'s share of the total
print("\nmean vs median (big gap = heavy tail):")
for c in ["gsc_impressions_total", "gsc_clicks_total"]:
    mean_v = feat[c].mean()
    median_v = feat[c].median()
    p99 = feat[c].quantile(0.99)
    top1pct_share = feat[feat[c] >= p99][c].sum() / feat[c].sum()
    print(f"  {c}: mean={mean_v:.1f}, median={median_v:.1f}, p99={p99:.1f}, "
          f"top 1% of rows hold {top1pct_share:.1%} of total {c}")

,count,mean,std,min,25%,50%,75%,max
gsc_impressions_total,176738.0,1587.986675,5431.337724,1.000000,20.00000,173.000000,1039.000000,617124.0
gsc_clicks_total,176738.0,4.650002,26.722649,0.000000,0.00000,0.000000,2.000000,5668.0
gsc_active_days,176738.0,20.431718,11.480153,1.000000,9.00000,26.000000,31.000000,31.0
gsc_avg_position_w,175304.0,16.722366,18.537978,0.019643,5.19444,8.568248,21.372440,309.0
gsc_ctr_x100,176738.0,0.459397,3.775992,0.000000,0.00000,0.000000,0.215796,100.0



mean vs median (big gap = heavy tail):
  gsc_impressions_total: mean=1588.0, median=173.0, p99=21799.8, top 1% of rows hold 25.1% of total gsc_impressions_total
  gsc_clicks_total: mean=4.7, median=0.0, p99=73.0, top 1% of rows hold 34.8% of total gsc_clicks_total


`gsc_impressions_total` and `gsc_clicks_total` are both heavily right-skewed. For impressions,
mean (1,588) is ~9× the median (173), and the top 1% of content items account for 25.1% of all
impressions. Clicks are more extreme still — the median is 0 (more than half of all 176,738
content items received zero clicks in March), and the top 1% of items hold 34.8% of all clicks.
`gsc_avg_position_w` and `gsc_active_days` are much closer to symmetric.

**Consequence for the signal tests below:** raw means on impressions/clicks would be dominated
by a small number of extreme-traffic pages and could mask or exaggerate real patterns. I use
medians (or log-transformed comparisons where a formal test needs it) rather than means when
comparing groups on impression/click-based metrics.

## 2. Signal test #1 / #2 / #3 (verdict each)

*Three safe signals, each with a mini-test and a verdict: CONFIRMED / OPPOSITE / MIXED / FALSE.*

Given the heavy-tailed distributions in §1, I use Spearman rank correlation (not Pearson) for
these tests — it's robust to the small number of extreme-traffic pages that would otherwise
dominate a linear correlation. Each test gets one verdict: CONFIRMED / OPPOSITE / MIXED / FALSE.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
staleness_meta = con.execute("""
SELECT content_hash_id, content_updated_date, content_created_date, word_count
FROM dim_content
""").fetchdf()

feat2 = feat.merge(staleness_meta, on="content_hash_id", how="left")
feat2["days_since_update"] = (pd.Timestamp("2026-03-31") - pd.to_datetime(feat2["content_updated_date"])).dt.days

print(f"rows with a valid days_since_update: {feat2['days_since_update'].notna().sum()} / {len(feat2)}")
feat2[["days_since_update", "word_count"]].describe()

rows with a valid days_since_update: 176738 / 176738


,days_since_update,word_count
count,176738.000000,121423.0
mean,-47.011163,2731.323612
std,44.024321,1178.982999
min,-97.000000,0.0
25%,-78.000000,2225.0
50%,-50.000000,2731.0
75%,-48.000000,3179.0
max,303.000000,29341.0


In [6]:
print(feat2["days_since_update"].value_counts(bins=10).sort_index())
print(f"\nrows with content_updated_date > content_created_date: "
      f"{(pd.to_datetime(feat2['content_updated_date']) > pd.to_datetime(feat2['content_created_date'])).sum()} / {len(feat2)}")
print(f"rows with content_updated_date == content_created_date: "
      f"{(pd.to_datetime(feat2['content_updated_date']) == pd.to_datetime(feat2['content_created_date'])).sum()}")

(-97.40100000000001, -57.0]    69029
(-57.0, -17.0]                 79694
(-17.0, 23.0]                    525
(23.0, 63.0]                   25670
(63.0, 103.0]                    234
(103.0, 143.0]                  1294
(143.0, 183.0]                    31
(183.0, 223.0]                   105
(223.0, 263.0]                   123
(263.0, 303.0]                    33
Name: count, dtype: int64

rows with content_updated_date > content_created_date: 176122 / 176738
rows with content_updated_date == content_created_date: 616


Before testing, one field is excluded from this section: `content_updated_date`. A diagnostic
found 84% of content clustering into two narrow future-dated buckets relative to March
(-97 to -17 days), which doesn't look like organic editorial activity — it looks like a
scheduled or synthetic timestamp, not real edit history. Using it as a "staleness" signal here
would test a false premise. This is flagged in §4 as a real limitation on Lane 2's staleness
assumption, not swept aside.

Given the heavy-tailed distributions in §1, I use Spearman rank correlation (robust to extreme
values) for continuous-vs-continuous tests, and median comparisons (not means) for grouped
tests. Each gets one verdict: CONFIRMED / OPPOSITE / MIXED / FALSE.

In [7]:
from scipy.stats import spearmanr

# Test 1: does a better (lower) position associate with a higher CTR?
valid = feat.dropna(subset=["gsc_avg_position_w", "gsc_ctr_x100"])
rho, pval = spearmanr(valid["gsc_avg_position_w"], valid["gsc_ctr_x100"])
print(f"Test 1 — position vs CTR (n={len(valid)}):")
print(f"  Spearman rho = {rho:.4f}, p = {pval:.2e}")
print(f"  (expect negative rho: better/lower position -> higher CTR)")

Test 1 — position vs CTR (n=175304):
  Spearman rho = -0.2456, p = 0.00e+00
  (expect negative rho: better/lower position -> higher CTR)


In [8]:
# Test 2: does more consistent presence (active days) associate with more total impressions?
rho2, pval2 = spearmanr(feat["gsc_active_days"], feat["gsc_impressions_total"])
print(f"Test 2 — active days vs impressions (n={len(feat)}):")
print(f"  Spearman rho = {rho2:.4f}, p = {pval2:.2e}")

Test 2 — active days vs impressions (n=176738):
  Spearman rho = 0.8700, p = 0.00e+00


In [9]:
# Test 3: does content_type associate with CTR? Use medians given the heavy tail.
by_type = feat.groupby("content_type")["gsc_ctr_x100"].agg(["median", "mean", "count"])
print("Test 3 — CTR by content_type (median, not mean, given the skew):")
display(by_type)

from scipy.stats import kruskal
groups = [g["gsc_ctr_x100"].values for _, g in feat.groupby("content_type")]
stat, pval3 = kruskal(*groups)
print(f"\nKruskal-Wallis (non-parametric group difference test): H = {stat:.2f}, p = {pval3:.2e}")

Test 3 — CTR by content_type (median, not mean, given the skew):


,median,mean,count
content_type,,,
comparison article,0.0,0.166188,3356
feedly article,0.0,2.136121,13476
keyword article,0.0,0.324245,159906



Kruskal-Wallis (non-parametric group difference test): H = 3820.38, p = 0.00e+00


**Test 1 — Position vs CTR.** Spearman rho = -0.2456 (p ≈ 0, n=175,304). Direction confirms the
SEO assumption (better position → higher CTR), but the relationship is weak — position alone
explains only a modest share of CTR variation.
**Verdict: CONFIRMED (weak).**

**Test 2 — Active days vs impressions.** Spearman rho = 0.8700 (p ≈ 0, n=176,738). Strong
positive relationship — content present more days accumulates more impressions.
**Verdict: CONFIRMED (strong)** — with a caveat: part of this strength is expected by
construction (more active days mechanically means more days to accumulate impressions across),
not purely a discovered behavioral pattern.

**Test 3 — Content type vs CTR.** Median CTR is 0.0 for all three content types — the typical
page of every type gets zero clicks. Means differ substantially (feedly 2.14% vs keyword
article 0.32%), and a Kruskal-Wallis test confirms the distributions differ significantly
(H=3820.38, p ≈ 0).
**Verdict: MIXED** — statistically real, but the practical takeaway is about tail behavior
(a minority of high-performing content per type), not a typical-page difference; reading this
as "content_type reliably predicts CTR" would overstate what the median says.

## 3. The flag-linked test

*Pick a signal one of FlyRank's real flags relies on. Does the data support the rule's assumption?*

**Flag tested:** `position_tier` — the bucketing rule from FlyRank's data dictionary
(top_3 ≤3, page_1 ≤10, striking ≤20, page_3_5 ≤50, deep >50), used throughout Lane 2's
framing (ML-02/03) to prioritize which pages matter most.

**Assumption being tested:** that these specific boundaries mark real, meaningful shifts in
performance (CTR) — not that CTR just declines smoothly with position, which would make the
tier boundaries somewhat arbitrary rather than load-bearing.

In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
def position_tier(pos):
    if pd.isna(pos): return "no_data"
    if pos <= 3: return "top_3"
    if pos <= 10: return "page_1"
    if pos <= 20: return "striking"
    if pos <= 50: return "page_3_5"
    return "deep"

feat3 = feat.copy()
feat3["position_tier"] = feat3["gsc_avg_position_w"].apply(position_tier)

tier_order = ["top_3", "page_1", "striking", "page_3_5", "deep", "no_data"]
by_tier = feat3.groupby("position_tier")["gsc_ctr_x100"].agg(["median", "mean", "count"]).reindex(tier_order)
print("CTR by position_tier:")
display(by_tier)

# Does CTR drop sharply at the tier boundaries, or smoothly across position generally?
# Compare: tier-based groups vs a simple continuous check (correlation, already have from Test 1: rho=-0.2456)
print(f"\nFor reference, continuous position-vs-CTR correlation (Test 1): rho = -0.2456")

# Kruskal-Wallis across the tiers (excluding no_data)
from scipy.stats import kruskal
groups3 = [g["gsc_ctr_x100"].values for name, g in feat3.groupby("position_tier") if name != "no_data"]
stat3, pval3 = kruskal(*groups3)
print(f"Kruskal-Wallis across tiers: H = {stat3:.2f}, p = {pval3:.2e}")

CTR by position_tier:


,median,mean,count
position_tier,,,
top_3,0.07485,1.003476,15043
page_1,0.00000,0.504287,83546
striking,0.00000,0.335375,30200
page_3_5,0.00000,0.238754,33123
deep,0.00000,0.091811,13392
no_data,0.00000,3.277714,1434



For reference, continuous position-vs-CTR correlation (Test 1): rho = -0.2456
Kruskal-Wallis across tiers: H = 9866.01, p = 0.00e+00


**Flag tested:** `position_tier` bucketing (top_3/page_1/striking/page_3_5/deep).

**Result:** Mean CTR steps down monotonically across every tier (top_3: 1.00% → deep: 0.09%),
and the tiers are statistically distinguishable (Kruskal-Wallis H=9866.01, p≈0) — the strongest
grouped effect found in this notebook. But median CTR is 0.0 for every tier except top_3 (0.075%)
— meaning the typical page in page_1 through deep experiences no measurable click behavior
difference; the mean gradient is driven by a minority of strong performers within each tier, not
a shift in what most content experiences. Separately, the `no_data` tier (1,434 items with no
valid position reading — see ML-04/05) shows the *highest* mean CTR of any group (3.28%), an
artifact of small-sample noise on unmeasured content, not real outperformance — a trap for any
downstream logic that doesn't handle this group explicitly.

**Verdict: MIXED.** Confirmed at the aggregate/mean level; not confirmed as a "typical page"
experience below `top_3`; the `no_data` group requires separate handling wherever this flag
is consumed.

## 4. What this means in practice

*Two or three sentences: what a content team should take from this.*

Position genuinely matters for clicks, but only near the very top — the real payoff shows up
almost entirely in the `top_3` tier; below that, most individual pages see essentially no clicks
regardless of whether they sit at position 8 or position 40, so a team chasing "page 1" as a
goal should know that page 1 alone rarely translates to real traffic for a typical page. Content
type is not a reliable lever on its own — the typical page of every type gets zero clicks, and
the differences that do exist come from a minority of standout articles, not from format choice
itself. Most importantly, the `content_updated_date` field this warehouse slice provides does
not behave like real editorial history (84% of content clusters into two narrow future-dated
batches relative to March) — any "staleness" or "needs refresh" decision should not be built on
this field as-is without first confirming with FlyRank what it actually tracks.

In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [✅] Every section above is filled — markdown thinking AND the code that backs it
- [✅] The notebook runs top to bottom with no errors (Runtime → Run all)
- [✅] No client names, URLs, or private queries anywhere
- [✅] My claims use careful words: observed, measured, directional, decision-support
- [✅] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.